In [0]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CFG = {
    "projected_prototypes_csv": "/path/to/exports/projected_prototypes.csv",
    "outcomes_table": "catalog.schema.step7_filtered_cohort_with_apos",
    "condition_table": "catalog.schema.condition_occurrence_cohort",
    "measurement_table": "catalog.schema.measurement_cohort",
    "drug_table": "catalog.schema.drug_exposure_cohort",
    "procedure_table": "catalog.schema.procedure_occurrence_cohort",  # ASSUMED -- verify column names
    "concept_table": "catalog.omop.concept",

    "classes_to_plot": ["gdm", "pe_ghtn"],

    "output_dir": "./prototype_visualizations_highres",
    "uc_output_dir": "/path/to/exports/prototype_visualizations_highres",

    "colors": {
        "condition": "#7C6FE0", "measurement": "#2E9E6B", "drug": "#D08A2E", "procedure": "#3B82C4",
        "background": "#FFFFFF", "text": "#1A1A1A", "muted": "#8A8A8A", "episode_marker": "#555555",
        "dx_marker": "#C0392B",
    },

    # conditions/procedures/drugs: None = no prevalence filtering at all.
    # Set to a number (0 or higher) to re-enable a raw-prevalence floor.
    "salience_threshold": {"condition": None, "procedure": None, "drug": None},

    # labs: OUT-OF-RANGE is the primary filter. Enrichment is an ADDITIONAL
    # "or" condition -- a lab with no range on file can still qualify if it's
    # notably more common in this class's population than the whole cohort.
    "measurement_enrichment_threshold": 1.5,
    "measurement_min_prevalence": 0.10,

    "tri_window_months": 3,  # local branch's trimester-delta window length, for x-axis window labeling

    # ALL exhaustive lists (every domain): bold any concept enriched >= this
    # ratio in its outcome class vs the whole cohort. 1.05 = "5% enrichment",
    # adjustable.
    "exhaustive_list_enrichment_threshold": 1.05,

    "max_label_chars": 75,  # patient-CARD labs only -- truncate longer labels; exhaustive list is untouched

    "save_dpi": 600,

    "card_procedure_fallback": True,
    "procedure_fallback_exclude": [],
}
os.makedirs(CFG["output_dir"], exist_ok=True)
os.makedirs(CFG["uc_output_dir"], exist_ok=True)

FULL_OUTCOME_NAMES = {
    "has_pe": "Preeclampsia",
    "has_inferred_pe": "Inferred Preeclampsia",
    "has_ghtn": "Gestational Hypertension",
    "has_gdm": "Gestational Diabetes Mellitus",
}
CLASS_DISPLAY_NAME = {
    "gdm": "Gestational Diabetes Mellitus",
    "pe_ghtn": "Preeclampsia / Gestational Hypertension",
}
DX_DATE_TO_OUTCOME = {
    # SHORT abbreviations -- this mapping is used ONLY for the on-plot "Dx:"
    "gdm_dx_date": "GDM",
    "pe_dx_date": "PE",
    "inferred_pe_dx_date": "PE",
    "ghtn_dx_date": "GHTN",
}
DOMAIN_DISPLAY_NAME = {"condition": "Conditions", "measurement": "Labs", "drug": "Medications", "procedure": "Procedures"}


def save_fig_everywhere(fig, filename, cfg):
    """Saves to BOTH local output_dir (quick look this session) and the
    UC Volume (persists/downloadable outside this cluster session).
    `filename` may include subdirectories (e.g. "gdm/global/foo.png") --
    those are created as needed under BOTH destinations. dpi now pulled
    from cfg["save_dpi"] instead of being hardcoded, so resolution is a
    single knob."""
    for out_dir in [cfg["output_dir"], cfg["uc_output_dir"]]:
        path = os.path.join(out_dir, filename)
        os.makedirs(os.path.dirname(path), exist_ok=True)
        fig.savefig(path, dpi=cfg["save_dpi"], bbox_inches="tight", facecolor=cfg["colors"]["background"])
    print(f"  saved {filename} to both {cfg['output_dir']} and {cfg['uc_output_dir']}")

## 1. Load projected prototypes, filter to GDM + PE/GHTN

In [0]:

proto_df = pd.read_csv(CFG["projected_prototypes_csv"])
proto_df = proto_df[proto_df["class_name"].isin(CFG["classes_to_plot"])].reset_index(drop=True)
proto_df["patient_id"] = proto_df["patient_id"].astype(str)
print(f"{len(proto_df)} prototypes to visualize ({proto_df.groupby(['branch','class_name']).size().to_dict()})")
proto_df.head(20)


## 2. Patient demographics + dx dates for every prototype patient

In [0]:

def load_patient_outcomes(patient_ids, cfg):
    id_list_sql = ", ".join(f"'{p}'" for p in set(patient_ids))
    df = spark.sql(f"""
        SELECT person_id, date_of_birth, recorded_episode_start, inferred_episode_start,
              has_pe, has_inferred_pe, has_ghtn, has_gdm, is_control,
              earliest_apo_dx_date, gdm_dx_date, pe_dx_date, inferred_pe_dx_date, ghtn_dx_date
        FROM {cfg['outcomes_table']}
        WHERE person_id IN ({id_list_sql})
    """).toPandas()
    df["person_id"] = df["person_id"].astype(str)
    date_cols = ["date_of_birth", "recorded_episode_start", "inferred_episode_start",
                "earliest_apo_dx_date", "gdm_dx_date", "pe_dx_date", "inferred_pe_dx_date", "ghtn_dx_date"]
    for c in date_cols:
        df[c] = pd.to_datetime(df[c])
    df["episode_start"] = df[["recorded_episode_start", "inferred_episode_start"]].min(axis=1, skipna=True)
    df["age"] = ((df["episode_start"] - df["date_of_birth"]).dt.days / 365.25).round().astype("Int64")
    return df.set_index("person_id")


def get_full_outcome_names(row):
    """PE and Inferred PE are DEDUPED into a single "Preeclampsia" entry --
    if both flags are true, the subtitle should say "Preeclampsia" once,
    not list it twice under two different labels."""
    names = []
    if row["has_pe"] == 1 or row["has_inferred_pe"] == 1:
        names.append("Preeclampsia")
    if row["has_ghtn"] == 1:
        names.append("Gestational Hypertension")
    if row["has_gdm"] == 1:
        names.append("Gestational Diabetes Mellitus")
    return names


def get_pe_ghtn_subtype(row):
    """Only meaningful for pe_ghtn-class prototypes -- returns which
    subtype(s) this patient actually has, for stratifying saved files
    into pe_ghtn/<branch>/<subtype>/ subfolders."""
    has_pe_flag = (row["has_pe"] == 1) or (row["has_inferred_pe"] == 1)
    has_ghtn_flag = (row["has_ghtn"] == 1)
    if has_pe_flag and has_ghtn_flag:
        return "both"
    elif has_pe_flag:
        return "pe_only"
    elif has_ghtn_flag:
        return "ghtn_only"
    else:
        return "unknown"  # shouldn't occur for a pe_ghtn-class prototype, but guarded


def get_dx_month_and_label(row):
    earliest = row["earliest_apo_dx_date"]
    if pd.isna(earliest) or pd.isna(row["episode_start"]):
        return None, None
    matched_label = None
    for col, label in DX_DATE_TO_OUTCOME.items():
        if pd.notna(row[col]) and row[col] == earliest:
            matched_label = label
            break
    months = (earliest.year - row["episode_start"].year) * 12 + (earliest.month - row["episode_start"].month)
    return months, matched_label


outcomes_df = load_patient_outcomes(proto_df["patient_id"].tolist(), CFG)
print(f"Loaded outcomes for {len(outcomes_df)} patients")


## 3. Per-class + overall population prevalence (for conditions/procedures/drugs threshold, and lab enrichment)

In [0]:

def _resolved_id_series(df, concept_col, source_col):
    """concept_id if valid (nonzero), else source_concept_id -- OMOP's
    *_concept_id is 0 when a source code wasn't mapped to a standard
    concept, with the real code living in *_source_concept_id instead."""
    cid = df[concept_col].fillna(0).astype(int)
    sid = df[source_col].fillna(0).astype(int)
    return cid.where(cid != 0, sid)


def population_prevalence_for_class(class_name, cfg, verbose=True):
    """class_name == "all" computes prevalence across the ENTIRE cohort --
    the denominator for lab enrichment (a lab drawn at similar rates in
    "all" vs. a specific outcome's population is routine, not distinguishing)."""
    cache_path = os.path.join(cfg["output_dir"], f"prevalence_{class_name}.csv")
    if os.path.exists(cache_path):
        if verbose:
            print(f"  loaded cached prevalence for {class_name} from {cache_path}")
        return pd.read_csv(cache_path)

    if class_name == "all":
        where = "1=1"
    elif class_name == "pe_ghtn":
        where = "has_pe = 1 OR has_inferred_pe = 1 OR has_ghtn = 1"
    elif class_name == "gdm":
        where = "has_gdm = 1"
    else:
        raise ValueError(f"unknown class_name {class_name!r}")

    pids_df = spark.sql(f"SELECT person_id FROM {cfg['outcomes_table']} WHERE {where}").toPandas()
    patient_ids = pids_df["person_id"].astype(str).tolist()
    n_total = len(patient_ids)
    id_list_sql = ", ".join(f"'{p}'" for p in patient_ids)

    rows = []
    for domain, table, concept_col, source_col in [
        ("condition", cfg["condition_table"], "condition_concept_id", "condition_source_concept_id"),
        ("procedure", cfg["procedure_table"], "procedure_concept_id", "procedure_source_concept_id"),
        ("drug", cfg["drug_table"], "drug_concept_id", "drug_concept_id"),
        ("measurement", cfg["measurement_table"], "measurement_concept_id", "measurement_source_concept_id"),
    ]:
        df = spark.sql(f"""
            SELECT person_id, {concept_col} AS concept_id, {source_col} AS source_concept_id
            FROM {table} WHERE person_id IN ({id_list_sql})
        """).toPandas()
        df["resolved_id"] = _resolved_id_series(df, "concept_id", "source_concept_id")
        agg = df.groupby("resolved_id")["person_id"].nunique().reset_index()
        agg.columns = ["concept_id", "n_patients"]
        agg["domain"] = domain
        agg["prevalence"] = agg["n_patients"] / n_total
        rows.append(agg)

    prevalence_df = pd.concat(rows, ignore_index=True)
    prevalence_df.to_csv(cache_path, index=False)
    if verbose:
        print(f"  computed prevalence for {class_name}: n_population={n_total}, "
              f"{len(prevalence_df)} distinct resolved concepts, saved to {cache_path}")
    return prevalence_df


prevalence_by_class = {name: population_prevalence_for_class(name, CFG) for name in CFG["classes_to_plot"]}
prevalence_by_class["all"] = population_prevalence_for_class("all", CFG)


## 4. Concept name lookup + per-patient data fetchers

In [0]:

def concept_names_for(concept_ids, cfg):
    concept_ids = [int(c) for c in set(concept_ids) if pd.notna(c) and int(c) != 0]
    if not concept_ids:
        return {}
    id_list_sql = ", ".join(f"'{c}'" for c in concept_ids)
    rows = spark.sql(f"""
        SELECT concept_id, concept_name FROM {cfg['concept_table']} WHERE concept_id IN ({id_list_sql})
    """).collect()
    return {int(r.concept_id): r.concept_name for r in rows}


def fetch_patient_domain(person_id, table, date_col, concept_col, source_concept_col, cfg,
                         value_col=None, unit_col=None):
    """No date filtering -- source _cohort tables are already pre-filtered
    upstream. Used for conditions/procedures/drugs; measurements use the
    dedicated fetch_measurements below instead."""
    extra_cols = f", {value_col}, {unit_col}" if value_col else ""
    df = spark.sql(f"""
        SELECT person_id, {concept_col} AS concept_id, {source_concept_col} AS source_concept_id,
              {date_col} AS event_date {extra_cols}
        FROM {table} WHERE person_id = '{person_id}'
    """).toPandas()
    if df.empty:
        # still add the columns downstream code expects (resolved_id,
        # concept_name) -- an empty df that's missing these columns causes
        # a KeyError the moment any filter/plot function references them,
        # even though "no events" should just mean "nothing to show"
        df["resolved_id"] = pd.Series(dtype="int64")
        df["concept_name"] = pd.Series(dtype="object")
        return df
    df["event_date"] = pd.to_datetime(df["event_date"])
    df = df.sort_values("event_date")
    df["resolved_id"] = _resolved_id_series(df, "concept_id", "source_concept_id")

    names = concept_names_for(set(df["resolved_id"][df["resolved_id"] != 0]), cfg)
    df["concept_name"] = df["resolved_id"].map(lambda x: names.get(int(x), f"Unmapped code ({int(x)})") if x != 0 else "Unmapped code")
    return df


def fetch_measurements(person_id, cfg):
    """Dedicated measurement fetcher -- pulls range_low/range_high (for
    out-of-range determination), value_source_value (fallback display
    when value_as_number is null), and unit_concept_id (joined to
    catalog.omop.concept for the actual unit NAME).

    FIX: unit_name is now built with .apply(..., axis=1) instead of
    .map() -- .map() only has access to ONE column's value per call, so
    the previous fallback branch (`else df["unit_source_value"]`)
    accidentally returned the ENTIRE Series object rather than that
    row's scalar value, corrupting the column (this is what leaked into
    the garbled text seen in the "wonky" output)."""
    df = spark.sql(f"""
        SELECT person_id, measurement_concept_id AS concept_id,
              measurement_source_concept_id AS source_concept_id,
              measurement_date AS event_date, value_as_number, value_source_value,
              unit_concept_id, unit_source_value, range_low, range_high
        FROM {cfg['measurement_table']} WHERE person_id = '{person_id}'
    """).toPandas()
    if df.empty:
        # same fix as fetch_patient_domain -- add every column downstream
        # code expects (resolved_id, concept_name, direction, display_label)
        # so an empty result doesn't KeyError later
        df["resolved_id"] = pd.Series(dtype="int64")
        df["concept_name"] = pd.Series(dtype="object")
        df["direction"] = pd.Series(dtype="object")
        df["display_label"] = pd.Series(dtype="object")
        return df
    df["event_date"] = pd.to_datetime(df["event_date"])
    df = df.sort_values("event_date")
    df["resolved_id"] = _resolved_id_series(df, "concept_id", "source_concept_id")

    concept_names = concept_names_for(set(df["resolved_id"][df["resolved_id"] != 0]), cfg)
    df["concept_name"] = df["resolved_id"].map(
        lambda x: concept_names.get(int(x), f"Unmapped code ({int(x)})") if x != 0 else "Unmapped code")

    unit_ids = set(df["unit_concept_id"].dropna().astype(int))
    unit_ids.discard(0)
    unit_names = concept_names_for(unit_ids, cfg)

    def resolve_unit_name(row):
        uid = row["unit_concept_id"]
        if pd.notna(uid) and int(uid) != 0:
            name = unit_names.get(int(uid))
            if name:
                return name
        usv = row["unit_source_value"]
        return usv if pd.notna(usv) else ""

    df["unit_name"] = df.apply(resolve_unit_name, axis=1)

    def determine_direction(row):
        if pd.notna(row["value_as_number"]) and pd.notna(row["range_low"]) and pd.notna(row["range_high"]):
            if row["value_as_number"] < row["range_low"]:
                return "down"
            elif row["value_as_number"] > row["range_high"]:
                return "up"
            else:
                return "normal"
        return None  # can't determine -- no numeric value or no range on file

    df["direction"] = df.apply(determine_direction, axis=1)

    def display_value(row):
        if pd.notna(row["value_as_number"]):
            unit = f" {row['unit_name']}" if row["unit_name"] else ""
            return f"{row['concept_name']}: {row['value_as_number']:.2g}{unit}"
        elif pd.notna(row["value_source_value"]):
            return f"{row['concept_name']}: {row['value_source_value']}"
        return row["concept_name"]

    df["display_label"] = df.apply(display_value, axis=1)
    return df


def fetch_all_domains(person_id, cfg):
    conditions = fetch_patient_domain(person_id, cfg["condition_table"], "condition_start_date",
                                      "condition_concept_id", "condition_source_concept_id", cfg)
    procedures = fetch_patient_domain(person_id, cfg["procedure_table"], "procedure_date",
                                      "procedure_concept_id", "procedure_source_concept_id", cfg)  # ASSUMED cols
    drugs = fetch_patient_domain(person_id, cfg["drug_table"], "drug_exposure_start_date",
                                 "drug_concept_id", "drug_concept_id", cfg)
    labs = fetch_measurements(person_id, cfg)
    return {"condition": conditions, "procedure": procedures, "drug": drugs, "measurement": labs}


## 5. Salience filtering (conditions/procedures/drugs) + lab filtering (out-of-range OR enrichment) + shared helpers

In [0]:

def filter_to_salient(domain_df, domain_name, class_name, threshold=None):
    """threshold=None -> no filtering. threshold=0 -> real filter (keeps
    only concepts with prevalence > 0 in this class's population)."""
    if domain_df.empty:
        return domain_df
    t = threshold if threshold is not None else CFG["salience_threshold"][domain_name]
    if t is None:
        return domain_df
    prev = prevalence_by_class[class_name]
    prev_domain = prev[prev["domain"] == domain_name].set_index("concept_id")["prevalence"]
    salient_mask = domain_df["resolved_id"].map(lambda x: prev_domain.get(int(x), 0) > t)
    return domain_df[salient_mask].reset_index(drop=True)


def filter_measurement_salient(measurement_df, class_name):
    """A lab is salient if EITHER:
      (a) it's OUTSIDE its documented normal range (direction is up/down), OR
      (b) it's ENRICHED -- notably more common in this class's population
          than the whole cohort, per CFG['measurement_enrichment_threshold']/
          CFG['measurement_min_prevalence'] -- catches labs with no range on
          file, or that are meaningful by frequency rather than one abnormal
          reading. This re-adds the enrichment logic from earlier, now
          combined (OR) with the out-of-range check rather than replacing it.
    Non-numeric (value_source_value only) results are kept unconditionally,
    same as before -- range comparison isn't possible for them."""
    if measurement_df.empty:
        return measurement_df

    out_of_range_mask = measurement_df["direction"].isin(["up", "down"])
    nonnumeric_mask = measurement_df["value_as_number"].isna() & measurement_df["value_source_value"].notna()

    class_prev = prevalence_by_class[class_name]
    class_prev = class_prev[class_prev["domain"] == "measurement"].set_index("concept_id")["prevalence"]
    all_prev = prevalence_by_class["all"]
    all_prev = all_prev[all_prev["domain"] == "measurement"].set_index("concept_id")["prevalence"]
    enrichment_threshold = CFG["measurement_enrichment_threshold"]
    min_prevalence = CFG["measurement_min_prevalence"]

    def is_enriched(resolved_id):
        cid = int(resolved_id)
        cp = class_prev.get(cid, 0)
        if cp < min_prevalence:
            return False
        ap = all_prev.get(cid, 1e-9)
        return (cp / ap) >= enrichment_threshold

    enrichment_mask = measurement_df["resolved_id"].map(is_enriched)

    keep_mask = out_of_range_mask | nonnumeric_mask | enrichment_mask
    return measurement_df[keep_mask].reset_index(drop=True)


def dedupe_earliest(domain_df):
    if domain_df.empty:
        return domain_df
    return domain_df.sort_values("event_date").drop_duplicates(subset="resolved_id", keep="first").reset_index(drop=True)


def date_to_month_index(event_date, episode_start):
    return (event_date.year - episode_start.year) * 12 + (event_date.month - episode_start.month)


def parse_local_window(window_name, tri_window_months=None):
    tri_window_months = tri_window_months or CFG["tri_window_months"]
    m = re.match(r"^tri_mo([+-]\d+)$", str(window_name))
    if not m:
        return None, None
    offset = int(m.group(1))
    return offset, offset + tri_window_months


In [0]:

TIER_HEIGHT = 0.9
ROW_GAP = 0.9
MIN_MONTH_GAP = 0.6  # scaled up alongside AXIS_TEXT_SIZE so boxes at the larger font
                     # don't overlap vertically/horizontally the way they did at TIER_HEIGHT=0.5

# UNIFIED text size -- concept box labels, x-axis tick labels, domain row
# labels, and episode/dx markers all use this SAME size, per request that
# "all text size in axes labels, and concept codes should be same and big"
AXIS_TEXT_SIZE = 28
FONT_SIZE_LABEL = AXIS_TEXT_SIZE  # kept as a separate name since it's referenced
                                  # elsewhere in this notebook, but now equals AXIS_TEXT_SIZE
CHAR_WIDTH_FACTOR = 0.62
LABEL_PADDING_IN = 0.25
FIG_WIDTH = 17
PLOT_AREA_FRACTION = 0.82


def estimate_label_width_months(label, inches_per_month):
    width_in = len(label) * FONT_SIZE_LABEL * CHAR_WIDTH_FACTOR / 72.0 + LABEL_PADDING_IN
    return width_in / inches_per_month


def assign_tiers(months, labels, inches_per_month, min_gap=MIN_MONTH_GAP):
    """LEFT-ALIGNED tier packing: a box starts AT its month and extends
    rightward by its estimated width, so the gap check compares the next
    item's start against the previous item's END (m + width) -- not a
    symmetric center-to-center gap, which was the right model for
    CENTERED text but is wrong now that boxes are left-justified to their
    date's dotted line."""
    order = np.argsort(months)
    widths = [estimate_label_width_months(l, inches_per_month) for l in labels]
    tier_last_end = []
    tier_of = np.empty(len(months), dtype=int)
    for idx in order:
        m = months[idx]; w = widths[idx]
        placed = False
        for t in range(len(tier_last_end)):
            if m - tier_last_end[t] >= min_gap:
                tier_last_end[t] = m + w
                tier_of[idx] = t
                placed = True
                break
        if not placed:
            tier_last_end.append(m + w)
            tier_of[idx] = len(tier_last_end) - 1
    return tier_of, max(len(tier_last_end), 1)


MARKER_FONT_SIZE = AXIS_TEXT_SIZE  # same unified size as everything else


def clamp_marker(true_month, win_start, win_end):
    """For LOCAL prototypes: if a marker's TRUE month (episode start = 0,
    or the Dx month) falls outside the displayed window, clamp its drawn
    position to the nearest window edge instead of letting it hang off
    to the side or stretch the x-axis -- then return a suffix noting the
    true month so it's still informative how far outside the window it
    actually occurred. For GLOBAL prototypes (win_start is None), this is
    a no-op -- returns the true month unchanged with no suffix."""
    if win_start is None:
        return true_month, ""
    if true_month < win_start:
        return win_start, f" (M{true_month})"
    elif true_month > win_end:
        return win_end, f" (M{true_month})"
    else:
        return true_month, ""


def light_opaque(hex_color, blend=0.85):
    """Blends a domain color toward white to produce a light, fully
    OPAQUE tint -- same hue family, but opaque (alpha=1.0) so nothing
    behind the box (like a dotted connector line) shows through it,
    while staying light enough for black text to stay readable on top."""
    h = hex_color.lstrip('#')
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    r = int(r + (255 - r) * blend); g = int(g + (255 - g) * blend); b = int(b + (255 - b) * blend)
    return f'#{r:02x}{g:02x}{b:02x}'


def truncate_label(label, max_chars=None):
    """Patient-CARD-only truncation -- exhaustive list always shows the
    full label untouched.

    If the label exceeds max_chars, first try to cut it at a natural
    break: search for " in " and " by " (with surrounding whitespace,
    so we don't match mid-word) and cut to whichever occurs EARLIER in
    the string. Only falls back to a plain end truncation if neither
    phrase is present at all."""
    max_chars = max_chars if max_chars is not None else CFG["max_label_chars"]
    if len(label) <= max_chars:
        return label
    cut_positions = [label.find(sep) for sep in (" in ", " by ") if label.find(sep) != -1]
    if cut_positions:
        cut_at = min(cut_positions)
        return label[:cut_at] + "..."
    return label[:max_chars - 3] + "..."


## 7. Visualization A1 — combined card (Conditions + Medications only; Procedures excluded)

In [0]:
def age_to_range_label(age):
    """Buckets an exact age into a 10-year range for display, e.g. 27 -> '20-29'.
    `age` comes from load_patient_outcomes' pandas nullable Int64 column, so it
    can legitimately be pd.NA (missing date_of_birth or episode_start) -- guard
    that explicitly rather than letting int(age) raise on NA."""
    if pd.isna(age):
        return "unknown"
    low = (int(age) // 10) * 10
    return f"{low}-{low + 9}"


def _card_header(fig, fig_height, prototype_idx, class_name, age, full_outcomes,
                 branch, window_name, c):
    branch_label = "Global" if branch == "global" else "Local"
    subtitle = f"Age {age_to_range_label(age)} \u00b7 {branch_label} Prototype"
    win_start, win_end = (None, None)
    if branch == "local" and window_name:
        win_start, win_end = parse_local_window(window_name)
        if win_start is not None:
            subtitle += f" (Window: M{win_start}\u2013M{win_end})"
    subtitle += f" \u00b7 {', '.join(full_outcomes)}"

    # domain_title (e.g. "Conditions & Medications" / "Labs") dropped from
    # the header entirely per request -- title is now just the prototype/class line
    fig.text(0.05, 1 - 0.55 / fig_height,
             f"Patient Prototype {prototype_idx} for {CLASS_DISPLAY_NAME[class_name]}",
             fontsize=35, fontweight="bold", color=c["text"], va="top")
    fig.text(0.05, 1 - 1.15 / fig_height, subtitle, fontsize=32, color=c["muted"], va="top")
    return win_start, win_end


def _restrict_to_local_window(df, episode_start, win_start, win_end):
    """For LOCAL prototypes: keep ONLY events falling inside the specific
    trimester-delta window this prototype was drawn from -- not the
    patient's whole available history the way a GLOBAL prototype's card
    shows. No shading is drawn for this anymore; the window is expressed
    purely by which events are present and by the x-axis range itself."""
    if df.empty or win_start is None:
        return df
    window_start_date = episode_start + pd.DateOffset(months=win_start)
    window_end_date = episode_start + pd.DateOffset(months=win_end)
    return df[(df["event_date"] >= window_start_date) & (df["event_date"] < window_end_date)].reset_index(drop=True)


def plot_combined_patient_card(prototype_idx, patient_id, class_name, cfg,
                               branch="global", window_name=None, subtype_folder=None,
                               threshold_overrides=None, save=True):
    """Conditions + Medications ONLY -- Procedures excluded from the
    patient card per your request (still shown in the exhaustive list).
    For LOCAL prototypes, restricts to only the events inside that
    prototype's specific trimester window (no shading -- the window is
    now expressed by what data is shown, and by the x-axis range)."""
    row = outcomes_df.loc[patient_id]
    age, episode_start = row["age"], row["episode_start"]
    full_outcomes = get_full_outcome_names(row)
    dx_month, dx_label = get_dx_month_and_label(row)

    domains = fetch_all_domains(patient_id, cfg)

    # resolve the local window FIRST (before filtering data), since
    # restriction needs win_start/win_end regardless of what gets drawn
    win_start, win_end = (None, None)
    if branch == "local" and window_name:
        win_start, win_end = parse_local_window(window_name)

    salient = {}
    for name in ["condition", "drug"]:
        t = (threshold_overrides or {}).get(name)
        df = domains[name]
        df = df[df["resolved_id"] != 0]
        if win_start is not None:
            df = _restrict_to_local_window(df, episode_start, win_start, win_end)
        df = filter_to_salient(df, name, class_name, threshold=t)
        salient[name] = dedupe_earliest(df)

    c = cfg["colors"]
    domain_order_bottom_up = ["drug", "condition"]
    row_label = {"condition": "Conditions", "drug": "Medications"}
    # text_fns = {"condition": lambda r: r["concept_name"], "drug": lambda r: r["concept_name"]}
    text_fns = {"condition": lambda r: r["concept_name"][:26], "drug": lambda r: r["concept_name"][:26]}

    active_domains = [d for d in domain_order_bottom_up if not salient[d].empty]

    if not active_domains:
        # nothing left after filtering -- don't produce a figure at all
        return None, None

    all_months, layout = [], {}
    for name in active_domains:
        df = salient[name]
        months = np.array([date_to_month_index(d, episode_start) for d in df["event_date"]])
        labels = [text_fns[name](r) for _, r in df.iterrows()]
        all_months.extend(months.tolist())
        layout[name] = {"df": df, "months": months, "labels": labels}

    if win_start is not None:
        # x-axis scoped to the window itself (plus small padding), rather
        # than extending out to episode start / dx if those fall outside it
        max_month = max(all_months + [win_end])
        min_month = min(all_months + [win_start])
    else:
        max_month = max(all_months + [dx_month or 0, 1])
        min_month = min(all_months + [0])

    month_span = max(max_month - min_month, 1)
    inches_per_month = (FIG_WIDTH * PLOT_AREA_FRACTION) / (month_span + 3)

    for name in active_domains:
        tiers, n_tiers = assign_tiers(layout[name]["months"], layout[name]["labels"], inches_per_month)
        layout[name]["tiers"], layout[name]["n_tiers"] = tiers, n_tiers

    baselines, cursor = {}, 0.4
    for name in active_domains:
        baselines[name] = cursor
        cursor += layout[name]["n_tiers"] * TIER_HEIGHT + ROW_GAP
    top_of_stack = cursor if active_domains else 1.0

    # fig_height = max(4.5, 1.6 + top_of_stack)
    # fig = plt.figure(figsize=(FIG_WIDTH, fig_height))
    # fig.patch.set_facecolor(c["background"])
    # _card_header(fig, fig_height, prototype_idx, class_name, "Conditions & Medications",
    #             age, full_outcomes, branch, window_name, c)
    fig_height = max(4.5, 1.6 + top_of_stack)
    fig = plt.figure(figsize=(FIG_WIDTH, fig_height))
    fig.patch.set_facecolor(c["background"])
    _card_header(fig, fig_height, prototype_idx, class_name,
                age, full_outcomes, branch, window_name, c)

    ax = fig.add_axes([0.08, 0.10, 0.90, 0.78])
    ax.set_facecolor(c["background"])

    ylim_bottom, ylim_top = -0.3, top_of_stack + 0.5

    for name in active_domains:
        d = layout[name]
        baseline = baselines[name]
        color = c[name]
        band_center = baseline + (d["n_tiers"] * TIER_HEIGHT) / 2
        ax.text(min_month - 0.4, band_center, row_label[name],
                ha="right", va="center", fontsize=AXIS_TEXT_SIZE, color=color, fontweight="bold")
        light_color = light_opaque(color)
        for i in range(len(d["months"])):
            m, tier, label = d["months"][i], d["tiers"][i], d["labels"][i]
            label_y = baseline + tier * TIER_HEIGHT + TIER_HEIGHT * 0.5
            # connector line now reaches the TRUE x-axis (ylim_bottom), not
            # just partway up from the row's baseline
            ax.plot([m, m], [ylim_bottom, label_y - 0.1], linestyle="--", linewidth=0.7, color=c["muted"], zorder=1)
            # LEFT-ALIGNED, OPAQUE box (light tint of the domain color, alpha=1.0
            # -- no longer translucent, so the dotted line can't show through it)
            ax.text(m, label_y, label, fontsize=FONT_SIZE_LABEL, color=c["text"], ha="left", va="center",
                    bbox=dict(boxstyle="round,pad=0.35", facecolor=light_color, alpha=1.0, edgecolor=color, linewidth=1.0), zorder=2)

    # episode start / dx: HORIZONTAL text, bottom-RIGHT of their line, bigger
    # font. For LOCAL prototypes, clamp_marker pins the line to the window
    # edge if the true month falls outside the displayed window (rather
    # than hanging off to the side), and appends "(M<true month>)" so you
    # can still see how far before/after the window it actually happened.
    ep_x, ep_suffix = clamp_marker(0, win_start, win_end)
    ax.plot([ep_x, ep_x], [ylim_bottom, ylim_top], color=c["episode_marker"], linewidth=1.3, linestyle=":", zorder=0)
    ax.text(ep_x, ylim_bottom + 0.05, f"Episode start{ep_suffix}", fontsize=MARKER_FONT_SIZE, color=c["episode_marker"],
            ha="left", va="bottom")
    if dx_month is not None:
        dx_x, dx_suffix = clamp_marker(dx_month, win_start, win_end)
        ax.plot([dx_x, dx_x], [ylim_bottom, ylim_top], color=c["dx_marker"], linewidth=1.3, linestyle=":", zorder=0)
        ax.text(dx_x, ylim_bottom + 0.05, (f"Dx: {dx_label}" if dx_label else "Dx") + dx_suffix, fontsize=MARKER_FONT_SIZE,
                color=c["dx_marker"], ha="left", va="bottom", fontweight="bold")

    ax.set_xlim(min_month - 1.5, max_month + 3)  # extra right padding -- left-aligned labels extend rightward
    tick_months = list(range(int(min_month), int(max_month) + 1))
    tick_labels = [f"M{m}" if m != dx_month else f"M{m} Dx" for m in tick_months]
    ax.set_xticks(tick_months)
    ax.set_xticklabels(tick_labels, fontsize=AXIS_TEXT_SIZE, rotation=45 if len(tick_months) > 15 else 0)

    ax.set_ylim(ylim_bottom, ylim_top)
    ax.set_yticks([])
    for spine in ["top", "right", "left"]:
        ax.spines[spine].set_visible(False)
    ax.spines["bottom"].set_color(c["muted"])

    if save:
        folder = f"{class_name}/{branch}/" + (f"{subtype_folder}/" if subtype_folder else "")
        save_fig_everywhere(fig, f"{folder}prototype_{prototype_idx}_{class_name}_combined_card.png", cfg)
    return fig, ax

## 8. Visualization A2 — separate labs card (out-of-range OR enrichment, left-aligned)

In [0]:
def plot_measurement_card(prototype_idx, patient_id, class_name, cfg,
                          branch="global", window_name=None, subtype_folder=None, save=True):
    row = outcomes_df.loc[patient_id]
    age, episode_start = row["age"], row["episode_start"]
    full_outcomes = get_full_outcome_names(row)
    dx_month, dx_label = get_dx_month_and_label(row)

    domains = fetch_all_domains(patient_id, cfg)
    df = domains["measurement"]
    df = df[df["resolved_id"] != 0]

    win_start, win_end = (None, None)
    if branch == "local" and window_name:
        win_start, win_end = parse_local_window(window_name)
        df = _restrict_to_local_window(df, episode_start, win_start, win_end)

    df = filter_measurement_salient(df, class_name)
    df = dedupe_earliest(df)

    if df.empty:
        # nothing left after filtering -- don't produce a figure at all
        return None, None

    c = cfg["colors"]
    color = c["measurement"]

    def build_card_label(r):
        """Truncates ONLY the concept name, never the value/unit -- a lab
        with an out-of-range arrow REQUIRES a real value_as_number, so
        this guarantees that value stays visible whenever an arrow is
        shown, rather than risking truncation cutting it off."""
        name = truncate_label(r["concept_name"])
        if pd.notna(r["value_as_number"]):
            unit = f" {r['unit_name']}" if r["unit_name"] else ""
            return f"{name}: {r['value_as_number']:.2g}{unit}"
        elif pd.notna(r["value_source_value"]):
            return f"{name}: {r['value_source_value']}"
        return name

    months = np.array([date_to_month_index(d, episode_start) for d in df["event_date"]])
    labels = [build_card_label(r) for _, r in df.iterrows()]
    directions = df["direction"].tolist()

    if win_start is not None:
        max_month = max(list(months) + [win_end])
        min_month = min(list(months) + [win_start])
    else:
        max_month = max(list(months) + [dx_month or 0, 1])
        min_month = min(list(months) + [0])

    month_span = max(max_month - min_month, 1)
    inches_per_month = (FIG_WIDTH * PLOT_AREA_FRACTION) / (month_span + 3)

    if len(months) > 0:
        tiers, n_tiers = assign_tiers(months, labels, inches_per_month)
    else:
        tiers, n_tiers = np.array([]), 1

    # baseline = 0.4
    # top_of_stack = baseline + n_tiers * TIER_HEIGHT + ROW_GAP
    # fig_height = max(4, 1.6 + top_of_stack)
    # fig = plt.figure(figsize=(FIG_WIDTH, fig_height))
    # fig.patch.set_facecolor(c["background"])
    # _card_header(fig, fig_height, prototype_idx, class_name, "Labs", age, full_outcomes, branch, window_name, c)
    baseline = 0.4
    top_of_stack = baseline + n_tiers * TIER_HEIGHT + ROW_GAP
    fig_height = max(4, 1.6 + top_of_stack)
    fig = plt.figure(figsize=(FIG_WIDTH, fig_height))
    fig.patch.set_facecolor(c["background"])
    _card_header(fig, fig_height, prototype_idx, class_name, age, full_outcomes, branch, window_name, c)

    ax = fig.add_axes([0.08, 0.10, 0.90, 0.78])
    ax.set_facecolor(c["background"])

    ylim_bottom, ylim_top = -0.3, top_of_stack + 0.5

    light_color = light_opaque(color)
    for i in range(len(months)):
        m, tier, label, direction = months[i], tiers[i], labels[i], directions[i]
        label_y = baseline + tier * TIER_HEIGHT + TIER_HEIGHT * 0.5
        # connector line now reaches the TRUE x-axis (ylim_bottom)
        ax.plot([m, m], [ylim_bottom, label_y - 0.1], linestyle="--", linewidth=0.7, color=c["muted"], zorder=1)
        arrow = " \u2191" if direction == "up" else (" \u2193" if direction == "down" else "")
        # LEFT-ALIGNED, OPAQUE box
        ax.text(m, label_y, label + arrow, fontsize=FONT_SIZE_LABEL, color=c["text"], ha="left", va="center",
                bbox=dict(boxstyle="round,pad=0.35", facecolor=light_color, alpha=1.0, edgecolor=color, linewidth=1.0), zorder=2)

    # episode start / dx: clamped to the window edge for LOCAL prototypes
    # (see clamp_marker) instead of hanging off to the side, with the
    # true month appended so you can see how far outside the window it was
    ep_x, ep_suffix = clamp_marker(0, win_start, win_end)
    ax.plot([ep_x, ep_x], [ylim_bottom, ylim_top], color=c["episode_marker"], linewidth=1.3, linestyle=":", zorder=0)
    ax.text(ep_x, ylim_bottom + 0.05, f"Episode start{ep_suffix}", fontsize=MARKER_FONT_SIZE, color=c["episode_marker"],
            ha="left", va="bottom")
    if dx_month is not None:
        dx_x, dx_suffix = clamp_marker(dx_month, win_start, win_end)
        ax.plot([dx_x, dx_x], [ylim_bottom, ylim_top], color=c["dx_marker"], linewidth=1.3, linestyle=":", zorder=0)
        ax.text(dx_x, ylim_bottom + 0.05, (f"Dx: {dx_label}" if dx_label else "Dx") + dx_suffix, fontsize=MARKER_FONT_SIZE,
                color=c["dx_marker"], ha="left", va="bottom", fontweight="bold")

    ax.set_xlim(min_month - 1.5, max_month + 3)
    tick_months = list(range(int(min_month), int(max_month) + 1))
    tick_labels = [f"M{m}" if m != dx_month else f"M{m} Dx" for m in tick_months]
    ax.set_xticks(tick_months)
    ax.set_xticklabels(tick_labels, fontsize=AXIS_TEXT_SIZE, rotation=45 if len(tick_months) > 15 else 0)

    ax.set_ylim(ylim_bottom, ylim_top)
    ax.set_yticks([])
    for spine in ["top", "right", "left"]:
        ax.spines[spine].set_visible(False)
    ax.spines["bottom"].set_color(c["muted"])

    if save:
        folder = f"{class_name}/{branch}/" + (f"{subtype_folder}/" if subtype_folder else "")
        save_fig_everywhere(fig, f"{folder}prototype_{prototype_idx}_{class_name}_measurement_card.png", cfg)
    return fig, ax



## 9. Visualization B — exhaustive per-domain lists (bold if enriched)

In [0]:

def get_enrichment_ratio(resolved_id, domain_name, class_name):
    """class prevalence / overall-cohort prevalence for this concept, in
    this domain. Used by the exhaustive list to bold enriched concepts --
    reuses the SAME prevalence_by_class tables already computed for
    conditions/procedures/drugs/measurements (Section 3), so no new
    queries are needed for any domain."""
    class_prev = prevalence_by_class[class_name]
    class_prev = class_prev[class_prev["domain"] == domain_name].set_index("concept_id")["prevalence"]
    all_prev = prevalence_by_class["all"]
    all_prev = all_prev[all_prev["domain"] == domain_name].set_index("concept_id")["prevalence"]
    cp = class_prev.get(int(resolved_id), 0)
    ap = all_prev.get(int(resolved_id), 1e-9)
    return cp / ap


def plot_domain_exhaustive_list(prototype_idx, patient_id, class_name, domain_name, domain_df, cfg,
                                branch="global", subtype_folder=None, is_measurement=False, save=True):
    domain_df = domain_df[domain_df["resolved_id"] != 0]
    deduped = dedupe_earliest(domain_df)
    n_rows = len(deduped)

    if n_rows == 0:
        # nothing left after dedup/filtering -- don't produce a figure at all
        return None

    FONT_SIZE_DATE = AXIS_TEXT_SIZE
    FONT_SIZE_LABEL = AXIS_TEXT_SIZE
    ROW_HEIGHT_INCHES = 0.42
    HEADER_INCHES = 1.1
    enrichment_threshold = cfg["exhaustive_list_enrichment_threshold"]

    fig_height = HEADER_INCHES + max(n_rows, 1) * ROW_HEIGHT_INCHES
    fig = plt.figure(figsize=(10, fig_height))
    c = cfg["colors"]
    fig.patch.set_facecolor(c["background"])

    display_name = DOMAIN_DISPLAY_NAME[domain_name]
    fig.text(0.05, 1 - 0.35 / fig_height, f"Prototype {prototype_idx} ({CLASS_DISPLAY_NAME[class_name]}) \u2014 {display_name}",
             fontsize=30, fontweight="bold", color=c["text"], va="top")
    fig.text(0.05, 1 - 0.68 / fig_height,
             f"{n_rows} distinct {display_name.lower()} (bold = enriched \u2265{(enrichment_threshold-1)*100:.0f}% "
             f"in {CLASS_DISPLAY_NAME[class_name]} vs. overall cohort)",
             fontsize=28, fontweight="bold", color=c["muted"], va="top")

    header_frac = HEADER_INCHES / fig_height
    ax = fig.add_axes([0.05, 0.02, 0.9, 1 - header_frac - 0.02])
    ax.set_facecolor(c["background"]); ax.axis("off"); ax.set_xlim(0, 1)

    if deduped.empty:
        ax.text(0.02, 0.9, "(none)", fontsize=FONT_SIZE_LABEL, color="#B0B0B0", style="italic", va="top")
    else:
        row_height = 1.0 / n_rows
        for i, r in deduped.iterrows():
            y = 1 - (i + 0.5) * row_height
            date_str = r["event_date"].strftime("%b %d, %Y")
            ax.text(0.02, y, date_str, fontsize=FONT_SIZE_DATE, color=c["muted"], va="center", family="monospace")
            label = r["display_label"] if is_measurement else r["concept_name"]  # NOT truncated -- exhaustive list stays full
            ratio = get_enrichment_ratio(r["resolved_id"], domain_name, class_name)
            weight = "bold" if ratio >= enrichment_threshold else "normal"
            ax.text(0.22, y, label, fontsize=FONT_SIZE_LABEL, color=c["text"], va="center", fontweight=weight)
            ax.plot([0.20, 0.205], [y, y], color=c[domain_name], linewidth=4.5, solid_capstyle="round")

    if save:
        folder = f"{class_name}/{branch}/" + (f"{subtype_folder}/" if subtype_folder else "")
        save_fig_everywhere(fig, f"{folder}prototype_{prototype_idx}_{class_name}_{domain_name}_full.png", cfg)
    return fig


## 11. Full run — generate and save everything

In [0]:

# only these prototype_idx values get generated/saved this run -- set to
# None to go back to running everything in proto_df
# CFG["prototype_idx_filter"] = [9, 28, 22, 16]
CFG["prototype_idx_filter"] = [26]

run_df = proto_df
if CFG["prototype_idx_filter"] is not None:
    run_df = proto_df[proto_df["prototype_idx"].isin(CFG["prototype_idx_filter"])].reset_index(drop=True)
    print(f"Filtered to prototype_idx {CFG['prototype_idx_filter']}: "
         f"{len(run_df)} rows ({run_df.groupby(['branch','class_name']).size().to_dict()})")

counts = {
    "global": {"total": 0, "combined_saved": 0, "measurement_saved": 0},
    "local": {"total": 0, "combined_saved": 0, "measurement_saved": 0},
}

for i, proto_row in run_df.iterrows():
    if i == 0:
        continue
    prototype_idx = proto_row["prototype_idx"]
    patient_id = proto_row["patient_id"]
    class_name = proto_row["class_name"]
    branch = proto_row["branch"]
    window_name = proto_row.get("window_name")

    # pe_ghtn only: stratify saved files further into pe_only/ghtn_only/both
    # subfolders. gdm has only one outcome, so no subtype stratification needed.
    subtype_folder = get_pe_ghtn_subtype(outcomes_df.loc[patient_id]) if class_name == "pe_ghtn" else None

    print(f"Prototype {prototype_idx} ({class_name}, {branch}"
         f"{', ' + subtype_folder if subtype_folder else ''}), patient {patient_id}")

    counts[branch]["total"] += 1

    fig_combined, _ = plot_combined_patient_card(prototype_idx, patient_id, class_name, CFG,
                                                 branch=branch, window_name=window_name,
                                                 subtype_folder=subtype_folder)
    if fig_combined is not None:
        plt.close(fig_combined)
        counts[branch]["combined_saved"] += 1
    else:
        print("  (combined card: nothing passed filtering -- skipped)")

    fig_measurement, _ = plot_measurement_card(prototype_idx, patient_id, class_name, CFG,
                                               branch=branch, window_name=window_name,
                                               subtype_folder=subtype_folder)
    if fig_measurement is not None:
        plt.close(fig_measurement)
        counts[branch]["measurement_saved"] += 1
    else:
        print("  (measurement card: nothing passed filtering -- skipped)")

    domains = fetch_all_domains(patient_id, CFG)
    for domain_name, domain_df in domains.items():
        fig_domain = plot_domain_exhaustive_list(
            prototype_idx, patient_id, class_name, domain_name, domain_df, CFG,
            branch=branch, subtype_folder=subtype_folder, is_measurement=(domain_name == "measurement")
        )
        if fig_domain is not None:
            plt.close(fig_domain)
        else:
            print(f"  ({domain_name} exhaustive list: no distinct concepts -- skipped)")

print(f"\nAll visualizations saved to {CFG['output_dir']} and {CFG['uc_output_dir']}, "
     f"stratified by outcome/branch (and, for pe_ghtn, subtype) subfolders "
     f"(e.g. gdm/local/, pe_ghtn/global/pe_only/, pe_ghtn/local/both/)")

print("\n=== Patient card save counts ===")
for branch in ["global", "local"]:
    total = counts[branch]["total"]
    print(f"{branch.capitalize()} prototypes: {total} total")
    print(f"  Combined card saved:    {counts[branch]['combined_saved']} / {total}")
    print(f"  Measurement card saved: {counts[branch]['measurement_saved']} / {total}")

In [0]:
import inspect
import pyarrow.dataset as ds
import pyarrow.compute as pc

CFG["diag_meds_path"] = "/path/to/exports/meds_output_spark/data"
# MEDS `table` values -> card domains. Print the raw value_counts first and
# adjust this map if your table names differ (I haven't seen them).
CFG["diag_table_to_domain"] = {
    "condition_occurrence": "condition",
    "drug_exposure": "drug",
    "measurement": "measurement",
    "procedure_occurrence": "procedure",
}

# Read this to see whether the fetch restricts by date, table, or a join.
print(inspect.getsource(fetch_all_domains))

meds = ds.dataset(CFG["diag_meds_path"], format="parquet")

for i, proto_row in run_df.iterrows():
    if i == 0:
        continue
    pid_key, pid_int = proto_row["patient_id"], int(proto_row["patient_id"])
    wname = proto_row["window_name"]
    win_start, win_end = parse_local_window(wname)
    episode_start = outcomes_df.loc[pid_key, "episode_start"]  # matched the extraction anchor earlier
    ws = episode_start + pd.DateOffset(months=win_start)
    we = episode_start + pd.DateOffset(months=win_end)

    raw = meds.to_table(columns=["subject_id", "time", "code", "table"],
                        filter=pc.field("subject_id") == pid_int).to_pandas()
    raw_win = raw[(raw["time"] >= ws) & (raw["time"] < we)]

    print(f"\n=== proto {proto_row['prototype_idx']} {wname}  window [{ws}, {we}) ===")
    print("raw MEDS rows in window by source table:", raw_win["table"].value_counts().to_dict())

    doms = fetch_all_domains(pid_key, CFG)
    for table_name, dom in CFG["diag_table_to_domain"].items():
        n_raw = int((raw_win["table"] == table_name).sum())
        d = doms.get(dom)
        if d is None:
            print(f"  {dom:12s} raw_in_window={n_raw:3d} | not returned by fetch_all_domains")
            continue
        in_win = d[(d["event_date"] >= ws) & (d["event_date"] < we)]
        n_mapped = int((in_win["resolved_id"] != 0).sum())
        print(f"  {dom:12s} raw_in_window={n_raw:3d} | fetched_in_window={len(in_win):3d} "
              f"| resolved!=0={n_mapped:3d} | fetched_total={len(d)}")

    # Raw condition/lab rows in the window, so you can eyeball what's missing
    for table_name in ["condition_occurrence", "measurement"]:
        sub = raw_win[raw_win["table"] == table_name]
        if len(sub):
            print(f"  raw {table_name} codes in window:")
            print(sub[["time", "code"]].to_string(index=False))

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

# All knobs for the rating plots live under CFG["ratings"] so both cells share one source of truth
# (and so adding a 5th reviewer = append to reviewer_record_ids + reviewer_colors, nothing else).
CFG["ratings"] = {
    "table": "catalog.schema.clinician_prototype_review",
    "reviewer_record_ids": [7, 9, 10, 11],   # record_id per reviewer; position here = "Reviewer N" label
    "rep_prefix": "representativeness",
    "act_prefix": "actionability",
    "likert_values": [1, 2, 3, 4, 5],
    "likert_labels": {1: "Strongly\nDisagree", 2: "Disagree", 3: "Neutral", 4: "Agree", 5: "Strongly\nAgree"},

    # Criterion colors (used by the box plots): same blue / purple as before.
    "rep_color": "#A6CAEC",
    "act_color": "#DED9EA",
    # Per-reviewer colors (used by the bar chart, where reviewers are the legend). First three are
    # unchanged from the previous version; the 4th is a new pastel orange chosen to stay distinct
    # from blue/purple/green at the same saturation.
    "reviewer_colors": ["#A6CAEC", "#DED9EA", "#B8E0C8", "#F5D5A8"],

    # Bar chart layout. Figure is wider than before (6 -> 9 in) and group width a bit larger so
    # 4 bars per Likert value don't get cramped; count font drops 18 -> 14 for the same reason.
    "bar_figsize": (9, 8),
    "bar_group_width": 0.65,
    "bar_count_fontsize": 14,
    "bar_out": "prototype_ratings_by_reviewer.png",
    "save_dpi": 800,
    # Columns to drop per record_id so every reviewer is scored on the same 39 prototypes.
    # record_id 11 has 41 prototype columns; the last two (_40, _41) are ignored for both criteria.
    "exclude_columns": {11: ["representativeness_40", "representativeness_41",
                             "actionability_40", "actionability_41"]},
}
R = CFG["ratings"]
REVIEWER_LABELS = [f"Reviewer {i+1}" for i in range(len(R["reviewer_record_ids"]))]

# ---- LOAD: one record_id per reviewer; each has suffixed columns (representativeness_1, _2, ...) ----
df_all = spark.table(R["table"]).toPandas()


def gather_scores(df, prefix, exclude=()):
    """Flatten all `prefix` / `prefix_*` columns for one reviewer into a single list of ints.
    Columns named in `exclude` are skipped."""
    matching_cols = [c for c in df.columns
                     if (c == prefix or c.startswith(prefix + "_")) and c not in exclude]
    if not matching_cols:
        return []
    return df[matching_cols].melt(value_name="score")["score"].dropna().astype(int).tolist()

rep_by_reviewer, act_by_reviewer = {}, {}
for rid, label in zip(R["reviewer_record_ids"], REVIEWER_LABELS):
    df_r = df_all[df_all["record_id"] == rid]
    excl = R["exclude_columns"].get(rid, [])
    rep_by_reviewer[label] = gather_scores(df_r, R["rep_prefix"], excl)
    act_by_reviewer[label] = gather_scores(df_r, R["act_prefix"], excl)
    print(f"record_id {rid} ({label}): {len(rep_by_reviewer[label])} representativeness ratings, "
          f"{len(act_by_reviewer[label])} actionability ratings")


# ---- BAR CHART: counts per Likert value, bars grouped by reviewer ----
def plot_criterion_by_reviewer(ax, scores_by_reviewer, title):
    reviewers = list(scores_by_reviewer.keys())
    n = len(reviewers)
    bar_width = R["bar_group_width"] / n
    x = np.arange(len(R["likert_values"])) * 0.75

    for i, reviewer in enumerate(reviewers):
        scores = scores_by_reviewer[reviewer]
        counts = [scores.count(v) for v in R["likert_values"]]
        offset = (i - (n - 1) / 2) * bar_width   # center the cluster on each Likert tick
        bars = ax.bar(x + offset, counts, width=bar_width, color=R["reviewer_colors"][i % len(R["reviewer_colors"])],
                      edgecolor="black", linewidth=0.7, label=reviewer)
        for rect, cnt in zip(bars, counts):
            if cnt > 0:   # skip "0" labels to reduce clutter
                ax.text(rect.get_x() + rect.get_width() / 2, rect.get_height() + 0.1, str(cnt),
                        ha="center", va="bottom", fontsize=R["bar_count_fontsize"], fontweight="bold")

    ax.set_xticks(x)
    ax.set_xticklabels([f"{v}\n{R['likert_labels'][v]}" for v in R["likert_values"]], fontsize=14)
    ax.set_xlim(x[0] - 0.5, x[-1] + 0.5)
    max_count = max((max((scores_by_reviewer[r].count(v) for r in reviewers), default=0)
                     for v in R["likert_values"]), default=1)
    ax.set_ylim(0, max_count + 2.5)
    ax.set_title(title, fontsize=20, fontweight="bold", pad=14)
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))   # counts are integers; avoid 2.5-style ticks
    ax.tick_params(axis="y", labelsize=18)
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)


fig, axes = plt.subplots(2, 1, figsize=R["bar_figsize"])
plot_criterion_by_reviewer(axes[0], rep_by_reviewer, "Representativeness")
plot_criterion_by_reviewer(axes[1], act_by_reviewer, "Actionability")

# Legend moved outside the axes (right of the top panel): with 4 reviewers it would otherwise sit on
# top of the bars. bbox_inches="tight" on save keeps it in the exported PNG.
axes[0].legend(fontsize=12, frameon=True, edgecolor="black", framealpha=1,
               loc="upper left", bbox_to_anchor=(1.01, 1.0))
axes[-1].set_xlabel("Likert Rating", fontsize=20, labelpad=10)
for ax in axes:
    ax.set_ylabel("Count", fontsize=20)

plt.tight_layout()
plt.savefig(R["bar_out"], dpi=R["save_dpi"], bbox_inches="tight")
print(f"saved {R['bar_out']}")

In [0]:
# Diverging stacked bars: per reviewer, % of ratings at each Likert value, with "Neutral" centered on 0
# so disagreement extends left and agreement extends right. Reuses rep_by_reviewer / act_by_reviewer
# and CFG["ratings"] from the bar-chart cell.
R.update({
    "div_figsize": (11, 6.5),
    # Diverging palette, light red -> grey -> blue: red = disagree, grey = neutral, blue = agree.
    # Intensity encodes strength (1 and 5 are the most saturated) so "strongly" vs "somewhat"
    # stays readable without the legend. Red/blue (rather than orange/blue) keeps the two sides
    # unambiguous and still reads as a single gradient.
    "div_colors": {1: "#E0736B", 2: "#F6BDB8", 3: "#E2E2E2", 4: "#9CC3E6", 5: "#3B82C4"},
    # "div_colors": {1: "#8E80CC", 2: "#C9C1E6", 3: "#EDEDED", 4: "#A6CAEC", 5: "#3B82C4"}, # purple gradient
    "div_bar_height": 0.62,       # fraction of the row each bar fills
    "div_min_label_pct": 8,       # skip the in-segment % label when a segment is narrower than this (it won't fit)
    "div_label_fontsize": 14,
    "div_out": "prototype_ratings_diverging.png",
})

LV = R["likert_values"]
neutral = 3   # the value centered on zero; assumes a 5-point scale with 3 in the middle


def diverging_panel(ax, scores_by_reviewer, title):
    reviewers = list(scores_by_reviewer.keys())
    max_extent = 0   # furthest any bar reaches from 0, used for symmetric x-limits
    for row, reviewer in enumerate(reviewers):
        scores = scores_by_reviewer[reviewer]
        n = len(scores)
        if n == 0:
            continue
        pct = {v: 100 * scores.count(v) / n for v in LV}

        # Left edge of each segment. Neutral straddles zero; disagree values stack leftward from the
        # neutral block's left edge (2 nearest the center, 1 outermost); agree values stack rightward.
        left = {}
        left[3] = -pct[3] / 2
        left[2] = left[3] - pct[2]
        left[1] = left[2] - pct[1]
        left[4] = pct[3] / 2
        left[5] = left[4] + pct[4]

        for v in LV:
            if pct[v] == 0:
                continue
            ax.barh(row, pct[v], left=left[v], height=R["div_bar_height"],
                    color=R["div_colors"][v], edgecolor="black", linewidth=0.7)
            if pct[v] >= R["div_min_label_pct"]:
                ax.text(left[v] + pct[v] / 2, row, f"{pct[v]:.0f}%", ha="center", va="center",
                        fontsize=R["div_label_fontsize"], fontweight="bold",
                        color="white" if v in (1, 5) else "black")   # white on the dark end colors for contrast
        max_extent = max(max_extent, -left[1], left[5] + pct[5])

    ax.axvline(0, color="black", linewidth=1.2, zorder=0)   # reference line through the middle of Neutral
    ax.set_yticks(range(len(reviewers)))
    # n is not shown: all reviewers now rate the same 39 prototypes (see CFG["ratings"]["exclude_columns"])
    ax.set_yticklabels(reviewers, fontsize=20)
    ax.invert_yaxis()   # Reviewer 1 on top
    ax.set_title(title, fontsize=20, fontweight="bold", pad=12)
    for spine in ["top", "right", "left"]:
        ax.spines[spine].set_visible(False)
    ax.tick_params(axis="y", length=0)
    return max_extent


fig, axes = plt.subplots(2, 1, figsize=R["div_figsize"], sharex=True)
extents = [diverging_panel(axes[0], rep_by_reviewer, "Representativeness"),
           diverging_panel(axes[1], act_by_reviewer, "Actionability")]

# Symmetric limits around 0 (shared across panels) so left/right extents are visually comparable
lim = np.ceil(max(extents) / 10) * 10 + 5
axes[1].set_xlim(-lim, lim)
ticks = np.arange(-np.floor(lim / 25) * 25, np.floor(lim / 25) * 25 + 1, 25)
axes[1].set_xticks(ticks)
# Both sides are percentages, so show absolute values (no negative signs on the disagree side)
axes[1].set_xticklabels([f"{abs(int(t))}%" for t in ticks], fontsize=16)
axes[1].set_xlabel("% of ratings", fontsize=20, labelpad=8)

# Legend: one patch per rating value, ordered 1 -> 5 to mirror the left -> right reading order
from matplotlib.patches import Patch
handles = [Patch(facecolor=R["div_colors"][v], edgecolor="black",
                 label=f"{v} \u2013 {R['likert_labels'][v].replace(chr(10), ' ')}") for v in LV]
fig.legend(handles=handles, loc="lower center", ncol=len(LV), fontsize=16, frameon=False,
           bbox_to_anchor=(0.5, -0.02))

plt.tight_layout(rect=(0, 0.06, 1, 1))   # leave room at the bottom for the legend
plt.savefig(R["div_out"], dpi=900, bbox_inches="tight")
print(f"saved {R['div_out']}")

In [0]:
# Diverging stacked bars: per reviewer, % of ratings at each Likert value, with "Neutral" centered on 0
# so disagreement extends left and agreement extends right. Reuses rep_by_reviewer / act_by_reviewer
# and CFG["ratings"] from the bar-chart cell.
R.update({
    "div_figsize": (11, 7.5),     # taller than before (was 6.5) so the panels keep their size with the 2-row legend
    # Diverging palette, light red -> grey -> blue: red = disagree, grey = neutral, blue = agree.
    # Intensity encodes strength (1 and 5 are the most saturated) so "strongly" vs "somewhat"
    # stays readable without the legend. Red/blue (rather than orange/blue) keeps the two sides
    # unambiguous and still reads as a single gradient.
    "div_colors": {1: "#E0736B", 2: "#F6BDB8", 3: "#E2E2E2", 4: "#9CC3E6", 5: "#3B82C4"},
    # "div_colors": {1: "#8E80CC", 2: "#C9C1E6", 3: "#EDEDED", 4: "#A6CAEC", 5: "#3B82C4"}, # purple gradient
    "div_bar_height": 0.62,       # fraction of the row each bar fills
    "div_min_label_pct": 8,       # skip the in-segment % label when a segment is narrower than this (it won't fit)
    "div_label_fontsize": 14,
    "div_legend_fontsize": 18,    # was 16
    "div_legend_ncol": 3,         # 3 columns x 2 rows for the 5 entries -> two lines
    "div_legend_bottom": 0.1,    # fraction of the figure height reserved for the legend (was 0.06 for one row)
    "div_out": "prototype_ratings_diverging.png",
})

LV = R["likert_values"]
neutral = 3   # the value centered on zero; assumes a 5-point scale with 3 in the middle


def diverging_panel(ax, scores_by_reviewer, title):
    reviewers = list(scores_by_reviewer.keys())
    max_extent = 0   # furthest any bar reaches from 0, used for symmetric x-limits
    for row, reviewer in enumerate(reviewers):
        scores = scores_by_reviewer[reviewer]
        n = len(scores)
        if n == 0:
            continue
        pct = {v: 100 * scores.count(v) / n for v in LV}

        # Left edge of each segment. Neutral straddles zero; disagree values stack leftward from the
        # neutral block's left edge (2 nearest the center, 1 outermost); agree values stack rightward.
        left = {}
        left[3] = -pct[3] / 2
        left[2] = left[3] - pct[2]
        left[1] = left[2] - pct[1]
        left[4] = pct[3] / 2
        left[5] = left[4] + pct[4]

        for v in LV:
            if pct[v] == 0:
                continue
            ax.barh(row, pct[v], left=left[v], height=R["div_bar_height"],
                    color=R["div_colors"][v], edgecolor="black", linewidth=0.7)
            if pct[v] >= R["div_min_label_pct"]:
                ax.text(left[v] + pct[v] / 2, row, f"{pct[v]:.0f}%", ha="center", va="center",
                        fontsize=R["div_label_fontsize"], fontweight="bold",
                        color="white" if v in (1, 5) else "black")   # white on the dark end colors for contrast
        max_extent = max(max_extent, -left[1], left[5] + pct[5])

    ax.axvline(0, color="black", linewidth=1.2, zorder=0)   # reference line through the middle of Neutral
    ax.set_yticks(range(len(reviewers)))
    # n is not shown: all reviewers now rate the same 39 prototypes (see CFG["ratings"]["exclude_columns"])
    ax.set_yticklabels(reviewers, fontsize=20)
    ax.invert_yaxis()   # Reviewer 1 on top
    ax.set_title(title, fontsize=20, fontweight="bold", pad=12)
    for spine in ["top", "right", "left"]:
        ax.spines[spine].set_visible(False)
    ax.tick_params(axis="y", length=0)
    return max_extent


fig, axes = plt.subplots(2, 1, figsize=R["div_figsize"], sharex=True)
extents = [diverging_panel(axes[0], rep_by_reviewer, "Representativeness"),
           diverging_panel(axes[1], act_by_reviewer, "Actionability")]

# Symmetric limits around 0 (shared across panels) so left/right extents are visually comparable
lim = np.ceil(max(extents) / 10) * 10 + 5
axes[1].set_xlim(-lim, lim)
ticks = np.arange(-np.floor(lim / 25) * 25, np.floor(lim / 25) * 25 + 1, 25)
axes[1].set_xticks(ticks)
# Both sides are percentages, so show absolute values (no negative signs on the disagree side)
axes[1].set_xticklabels([f"{abs(int(t))}%" for t in ticks], fontsize=16)
axes[1].set_xlabel("% of ratings", fontsize=20, labelpad=8)

# Legend: one patch per rating value. Matplotlib fills legend columns top-to-bottom, so to read
# 1 2 3 / 4 5 across the two rows the handles are interleaved (1, 4 | 2, 5 | 3, blank), with an
# invisible spacer patch filling the empty sixth slot.
from matplotlib.patches import Patch
by_v = {v: Patch(facecolor=R["div_colors"][v], edgecolor="black",
                 label=f"{v} \u2013 {R['likert_labels'][v].replace(chr(10), ' ')}") for v in LV}
spacer = Patch(facecolor="none", edgecolor="none", label=" ")
handles = [by_v[1], by_v[4], by_v[2], by_v[5], by_v[3], spacer]
fig.legend(handles=handles, loc="lower center", ncol=R["div_legend_ncol"], fontsize=R["div_legend_fontsize"],
           frameon=False, columnspacing=1.8, handlelength=1.6, bbox_to_anchor=(0.5, -0.02))

plt.tight_layout(rect=(0, R["div_legend_bottom"], 1, 1))   # leave room at the bottom for the two-row legend
plt.savefig(R["div_out"], dpi=900, bbox_inches="tight")
print(f"saved {R['div_out']}")

In [0]:
from itertools import combinations
import warnings
from sklearn.metrics import cohen_kappa_score

R.update({
    "kappa_weights": "quadratic",     # "linear" or None (unweighted) also work; quadratic is the usual choice for ordinal Likert data
    # Camps read off the diverging-bar plot (post hoc / exploratory). Edit freely; pairs not listed are "across-camp".
    "camps": [("Reviewer 1", "Reviewer 3"), ("Reviewer 2", "Reviewer 4")],
    "n_boot": 2000,                   # bootstrap resamples of prototypes for the CIs
    "boot_seed": 0,                   # fixed so reported CIs are reproducible
    "ci_level": 95,
})

LV = R["likert_values"]
CRITERIA = {"Representativeness": R["rep_prefix"], "Actionability": R["act_prefix"]}


def rating_matrix(prefix):
    """Prototype x reviewer DataFrame, aligned by prototype number, NOT by list position, so a missing rating
    can't silently shift one reviewer's ratings against another's.
    The unsuffixed column (e.g. `representativeness`) is treated as prototype 1 (the first instance has no
    suffix; repeated instances are _2, _3, ...). Takes the first non-null per column in case a record_id spans
    several rows (e.g. repeated-instrument exports)."""
    cols = {}
    for rid, lab in zip(R["reviewer_record_ids"], REVIEWER_LABELS):
        df_r = df_all[df_all["record_id"] == rid]
        excl = R["exclude_columns"].get(rid, [])   # same exclusions as the plots (rid 11: _40, _41)
        keep = [c for c in df_r.columns
                if (c == prefix or c.startswith(prefix + "_")) and c not in excl]
        vals = df_r[keep].bfill().iloc[0]
        vals.index = [1 if c == prefix else int(c[len(prefix) + 1:]) for c in keep]   # unsuffixed -> prototype 1
        assert vals.index.is_unique, f"{lab}: prototype numbering collides (is there also a `{prefix}_1` column?)"
        cols[lab] = vals
    return pd.DataFrame(cols).sort_index().dropna()   # dropna: score only prototypes every reviewer rated

def wkappa(a, b):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")   # kappa is undefined (nan) if a resample has no variation; handled by nan-aware percentiles
        return cohen_kappa_score(a, b, labels=LV, weights=R["kappa_weights"])


pairs = list(combinations(REVIEWER_LABELS, 2))
camp_pairs = {frozenset(p) for p in R["camps"]}
rng = np.random.default_rng(R["boot_seed"])
lo_q, hi_q = (100 - R["ci_level"]) / 2, 100 - (100 - R["ci_level"]) / 2

results = {}
for crit, prefix in CRITERIA.items():
    M = rating_matrix(prefix).astype(int)
    n = len(M)
    X = {lab: M[lab].values for lab in REVIEWER_LABELS}

    # point estimates per pair
    rows = []
    for a, b in pairs:
        rows.append({"pair": f"{a} vs {b}", "camp": "within" if frozenset((a, b)) in camp_pairs else "across",
                     "kappa": wkappa(X[a], X[b]),
                     "exact_%": 100 * np.mean(X[a] == X[b]),
                     "within1_%": 100 * np.mean(np.abs(X[a] - X[b]) <= 1)})
    tab = pd.DataFrame(rows)

    # bootstrap over prototypes: resample prototypes (rows) with replacement, keeping each prototype's 4 ratings together
    boot = np.empty((R["n_boot"], len(pairs)))
    for i in range(R["n_boot"]):
        idx = rng.integers(0, n, n)
        boot[i] = [wkappa(X[a][idx], X[b][idx]) for a, b in pairs]
    tab["ci_lo"] = np.nanpercentile(boot, lo_q, axis=0)
    tab["ci_hi"] = np.nanpercentile(boot, hi_q, axis=0)

    # summary rows: mean kappa over all pairs (Light's kappa), within-camp pairs, across-camp pairs.
    # Summary CIs average the SAME resample across pairs, so they respect the correlation between pairs.
    def summarize(name, mask):
        b = np.nanmean(boot[:, mask], axis=1)
        return {"pair": name, "camp": "", "kappa": tab.loc[mask, "kappa"].mean(),
                "exact_%": tab.loc[mask, "exact_%"].mean(), "within1_%": tab.loc[mask, "within1_%"].mean(),
                "ci_lo": np.nanpercentile(b, lo_q), "ci_hi": np.nanpercentile(b, hi_q)}
    allm, wm, am = (np.ones(len(tab), bool), (tab["camp"] == "within").values, (tab["camp"] == "across").values)
    tab = pd.concat([tab, pd.DataFrame([summarize("MEAN, all pairs (Light's kappa)", allm),
                                        summarize("MEAN, within-camp pairs", wm),
                                        summarize("MEAN, across-camp pairs", am)])], ignore_index=True)
    results[crit] = tab
    print(f"\n=== {crit}  (n = {n} prototypes, {R['kappa_weights']}-weighted kappa, {R['ci_level']}% bootstrap CI) ===")
    print(tab.round({"kappa": 2, "exact_%": 0, "within1_%": 0, "ci_lo": 2, "ci_hi": 2}).to_string(index=False))

In [0]:
# Per-reviewer rating tendency, to quote in the response in place of the figure
for crit, d in [("Representativeness", rep_by_reviewer), ("Actionability", act_by_reviewer)]:
    print(crit)
    for lab, s in d.items():
        s = np.array(s)
        print(f"  {lab}: mean {s.mean():.2f}, % rated 1-2: {100*np.mean(s <= 2):.0f}, % rated 4-5: {100*np.mean(s >= 4):.0f}")

In [0]:
# %sql
# SELECT *
# FROM catalog.schema.clinician_prototype_review
# WHERE record_id IN (7, 9, 10, 11)
# LIMIT 5

# -- which columns have values of >= 5 for at least 3 of these rows
import re
import pandas as pd

CFG = {
    "table": "catalog.schema.clinician_prototype_review",
    "id_col": "record_id",
    "record_ids": [7, 9, 10, 11],
    "col_pattern": r"^(representativeness|actionability)",  # prefix match, case-insensitive; anything may follow
    "threshold": 4,                  # a cell counts if its value is >= this
    "min_rows": 3,                   # keep columns that meet the threshold in at least this many rows
}

# No LIMIT: the counts must see every matching row
id_list = ", ".join(str(i) for i in CFG["record_ids"])
df = spark.sql(f"SELECT * FROM {CFG['table']} WHERE {CFG['id_col']} IN ({id_list})").toPandas()
print(f"{len(df)} rows, {df[CFG['id_col']].nunique()} distinct {CFG['id_col']}")

# Restrict to the rating columns of interest
rating_cols = [c for c in df.columns if re.match(CFG["col_pattern"], c, flags=re.IGNORECASE)]
print(f"{len(rating_cols)} columns matched: {rating_cols}")
assert rating_cols, "no columns matched col_pattern; check the column names printed by df.columns"

# Coerce to numeric so ratings stored as strings ("5") still compare correctly;
# text values become NaN, and NaN >= 5 is False, so missing/non-numeric cells never count
num = df[rating_cols].apply(pd.to_numeric, errors="coerce")
ge = num >= CFG["threshold"]

res = pd.DataFrame({
    "n_rows_ge": ge.sum(),                                    # rows meeting the threshold
    "n_ids_ge": ge.groupby(df[CFG["id_col"]]).any().sum(),    # distinct record_ids with >= 1 qualifying row
})
res_kept = res[res["n_rows_ge"] >= CFG["min_rows"]].sort_values("n_rows_ge", ascending=False)

print(f"\nColumns with >= {CFG['threshold']} in at least {CFG['min_rows']} rows:")
print(res_kept.to_string() if len(res_kept) else "none")

# all matched columns, so you can see near-misses (for example 2 of the 3 required rows)
print("\nAll matched columns:")
print(res.sort_values("n_rows_ge", ascending=False).to_string())

# raw values for the matched columns, for an eyeball check
print(df[[CFG["id_col"]] + rating_cols].to_string(index=False))

In [0]:
import re
import pandas as pd

CFG = {
    "table": "catalog.schema.clinician_prototype_review",
    "id_col": "record_id",
    "record_ids": [7, 9, 10, 11],
    "col_pattern": r"^(representativeness|actionability)",  # prefix match to select candidate columns, case-insensitive
    "threshold": 4,                  # a cell counts if its value is >= this (same as your pasted cell)
    "max_rows_ge": 0,                # keep columns with at most this many qualifying rows (0 = none)
    "min_rated": 1,                  # require at least this many numeric ratings, so empty/free-text columns are not reported
    "branch": "local",               # keep only this branch
    # review column number -> (outcome, branch, patient prototype number shown on the cards)
    "proto_map": {
        # PE / GHTN
        1: ("pe_ghtn", "global", 17), 2: ("pe_ghtn", "local", 8),  4: ("pe_ghtn", "local", 9),
        5: ("pe_ghtn", "global", 20), 6: ("pe_ghtn", "global", 22), 7: ("pe_ghtn", "local", 6),
        8: ("pe_ghtn", "local", 10), 9: ("pe_ghtn", "global", 18), 10: ("pe_ghtn", "global", 19),
        11: ("pe_ghtn", "global", 21), 12: ("pe_ghtn", "global", 23), 13: ("pe_ghtn", "global", 24),
        14: ("pe_ghtn", "global", 25), 15: ("pe_ghtn", "local", 7),
        # GDM
        3: ("gdm", "global", 26), 16: ("gdm", "global", 27), 17: ("gdm", "global", 28),
        18: ("gdm", "global", 29), 19: ("gdm", "global", 30), 20: ("gdm", "global", 31),
        21: ("gdm", "global", 32), 22: ("gdm", "global", 33), 23: ("gdm", "global", 34),
        24: ("gdm", "local", 12), 25: ("gdm", "local", 13), 26: ("gdm", "local", 14),
        27: ("gdm", "local", 15), 28: ("gdm", "local", 16), 29: ("gdm", "local", 17),
        30: ("gdm", "local", 18), 31: ("gdm", "local", 19), 32: ("gdm", "local", 20),
        33: ("gdm", "local", 22), 34: ("gdm", "local", 23), 35: ("gdm", "local", 24),
        36: ("gdm", "local", 25), 37: ("gdm", "local", 26), 38: ("gdm", "local", 27),
        39: ("gdm", "local", 28),
    },
}

# No LIMIT: the counts must see every matching row
id_list = ", ".join(str(i) for i in CFG["record_ids"])
df = spark.sql(f"SELECT * FROM {CFG['table']} WHERE {CFG['id_col']} IN ({id_list})").toPandas()
print(f"{len(df)} rows, {df[CFG['id_col']].nunique()} distinct {CFG['id_col']}")

rating_cols = [c for c in df.columns if re.match(CFG["col_pattern"], c, flags=re.IGNORECASE)]
assert rating_cols, "no columns matched col_pattern; check df.columns"

# Coerce to numeric so string ratings ("5") compare correctly; text becomes NaN, and NaN >= threshold is False
num = df[rating_cols].apply(pd.to_numeric, errors="coerce")
ge = num >= CFG["threshold"]

res = pd.DataFrame({
    "n_rows_ge": ge.sum(),                                    # rows meeting the threshold
    "n_ids_ge": ge.groupby(df[CFG["id_col"]]).any().sum(),    # distinct record_ids with >= 1 qualifying row
    "n_rated": num.notna().sum(),                             # rows with any numeric rating in this column
}).rename_axis("column").reset_index()

# Strict column-name parse: <rating> or <rating>_<number>. The bare name (no number) is prototype 1.
# Anything else (e.g. representativeness_comment) fails to parse and is reported below, not guessed at.
PARSE = re.compile(r"^(representativeness|actionability)(?:_(\d+))?$", re.IGNORECASE)


def parse_col(col):
    m = PARSE.match(col.strip())
    if not m:
        return pd.Series({"rating": None, "review_proto": None})
    return pd.Series({"rating": m.group(1).lower(), "review_proto": int(m.group(2)) if m.group(2) else 1})


res[["rating", "review_proto"]] = res["column"].apply(parse_col)
res["review_proto"] = res["review_proto"].astype("Int64")   # nullable int so unparsed names stay <NA>

pm = (pd.DataFrame.from_dict(CFG["proto_map"], orient="index", columns=["outcome", "branch", "patient_prototype"])
      .rename_axis("review_proto").reset_index())
pm["review_proto"] = pm["review_proto"].astype("Int64")
res = res.merge(pm, on="review_proto", how="left")

# A column that failed to parse, or parsed to a number missing from proto_map, is invisible to the branch filter,
# so report it instead of dropping it silently
unmapped = res[res["branch"].isna()]
if len(unmapped):
    print(f"WARNING: {len(unmapped)} matched columns did not map to a prototype:\n{unmapped[['column', 'review_proto']].to_string(index=False)}")

# Two columns resolving to the same (rating, prototype), e.g. both 'representativeness' and 'representativeness_1'
dups = res[res.duplicated(["rating", "review_proto"], keep=False) & res["review_proto"].notna()]
if len(dups):
    print(f"WARNING: duplicate (rating, prototype) columns:\n{dups[['column', 'review_proto']].to_string(index=False)}")

# Parse check: column name -> review number -> mapping (eyeball this once)
print(res[["column", "review_proto", "outcome", "branch", "patient_prototype"]].to_string(index=False))

hits = res[(res["branch"] == CFG["branch"])
           & (res["n_rows_ge"] <= CFG["max_rows_ge"])
           & (res["n_rated"] >= CFG["min_rated"])
           ].sort_values(["outcome", "patient_prototype", "rating"])

print(f"\n{CFG['branch']} prototype columns with <= {CFG['max_rows_ge']} ratings >= {CFG['threshold']}:")
print(hits[["outcome", "patient_prototype", "rating", "column", "n_rows_ge", "n_ids_ge", "n_rated"]].to_string(index=False)
      if len(hits) else "none")